# Idea Generation Template Playground

Experiment with `idea_generation.jinja2` across different research areas and models.

**Template variable:** `{{ area }}` — a free-text research area description.

## Setup

In [61]:
import sys
import os
import re
from pathlib import Path
from jinja2 import Template
from IPython.display import display, HTML, Markdown
import toml

# ── paths ─────────────────────────────────────────────────────────────────────
# Notebook lives at: src/novelty_eval/iclr_test/
SRC_ROOT  = Path("../..").resolve()   # → .../Academic-Collabs/src
REPO_ROOT = SRC_ROOT.parent           # → .../Academic-Collabs
sys.path.insert(0, str(SRC_ROOT))

# ── pre-load API keys BEFORE importing utils (it inits clients at module level) ──
_secrets_path = REPO_ROOT / "secrets.toml"
assert _secrets_path.exists(), f"secrets.toml not found at {_secrets_path}"
_secrets = toml.load(_secrets_path)

os.environ["SECRETS"]          = str(_secrets_path)
os.environ["OUTPUT_DIR"]       = str(REPO_ROOT / "output")
os.environ["OPENAI_API_KEY"]   = _secrets.get("openai_key", "")
os.environ["ANTHROPIC_API_KEY"] = _secrets.get("anthropic", "")

from utils import prompt_openai_client   # handles both OpenAI and Claude

TEMPLATE_PATH = Path("templates/idea_generation_3.jinja2")
TEMPLATE = Template(TEMPLATE_PATH.read_text())

print(f"SRC_ROOT : {SRC_ROOT}")
print(f"REPO_ROOT: {REPO_ROOT}")
print(f"Template : {TEMPLATE_PATH.resolve()}")

SRC_ROOT : /Users/noysternlicht/myCode/Academic-Collabs/src
REPO_ROOT: /Users/noysternlicht/myCode/Academic-Collabs
Template : /Users/noysternlicht/myCode/Academic-Collabs/src/comparative_idea_quality_eval/iclr_test/templates/idea_generation_3.jinja2


---
## Configuration

Edit the two cells below to choose your areas and models.

In [62]:
# ── Research areas to probe ──────────────────────────────────────────────────
AREAS = [
    "foundation or frontier models, including LLMs",
    "foundation or frontier models, including LLMs",
    "foundation or frontier models, including LLMs",
]

In [63]:
# ── Models to compare ────────────────────────────────────────────────────────
MODELS = [
    "claude-sonnet-4-5",
]

---
## Generate ideas

In [64]:
def parse_abstract(raw: str) -> str:
    """Extract abstract text from model output.

    Handles:
      - abstract: "..."          (quoted, as instructed)
      - **abstract**: text       (markdown bold label, model drift)
      - abstract: text           (unquoted)
    Strips surrounding markdown bold markers (**) from the extracted text.
    """
    patterns = [
        r'(?:\*\*)?abstract(?:\*\*)?\s*:\s*[""](.+?)[""]\s*$',   # quoted (straight or curly)
        r'(?:\*\*)?abstract(?:\*\*)?\s*:\s*(.+)',                  # unquoted / bold
    ]
    for pat in patterns:
        m = re.search(pat, raw, re.DOTALL | re.IGNORECASE)
        if m:
            text = m.group(1).strip()
            text = re.sub(r'^\*+\s*', '', text)   # strip leading asterisks
            text = re.sub(r'\s*\*+$', '', text)   # strip trailing asterisks
            return text
    return raw.strip()


def generate_idea(area: str, model: str) -> str:
    prompt = TEMPLATE.render(area=area)
    raw = prompt_openai_client(prompt, engine=model, max_completion_tokens=512)
    return parse_abstract(raw)


# results[area][model] = list of abstract strings (one per occurrence of area in AREAS)
results: dict[str, dict[str, list[str]]] = {}

total = len(AREAS) * len(MODELS)
done  = 0

for area in AREAS:
    if area not in results:
        results[area] = {m: [] for m in MODELS}
    for model in MODELS:
        done += 1
        print(f"[{done}/{total}]  {model}  |  {area[:60]}...")
        results[area][model].append(generate_idea(area, model))

print("\nDone!")

[1/3]  claude-sonnet-4-5  |  foundation or frontier models, including LLMs...
[2/3]  claude-sonnet-4-5  |  foundation or frontier models, including LLMs...
[3/3]  claude-sonnet-4-5  |  foundation or frontier models, including LLMs...

Done!


---
## Results

In [65]:
# ── colour palette (one colour per model) ────────────────────────────────────
PALETTE = [
    "#e8f4f8",   # light blue
    "#f0f8e8",   # light green
    "#fdf3e7",   # light orange
    "#f8e8f4",   # light purple
    "#e8f8f0",   # light teal
]
MODEL_COLORS = {m: PALETTE[i % len(PALETTE)] for i, m in enumerate(MODELS)}


def render_results(results: dict, models: list[str]) -> None:
    for area, model_outputs in results.items():
        display(HTML(
            f"""
            <div style="
                background:#1a1a2e; color:#eaeaea;
                padding:14px 20px; border-radius:8px;
                font-family:sans-serif; font-size:18px;
                margin-top:32px; margin-bottom:8px;
            ">
                <span style="opacity:.6; font-size:16px;">Research Area</span><br/>
                <b>{area}</b>
            </div>
            """
        ))

        for model in models:
            abstracts = model_outputs.get(model, [])
            bg = MODEL_COLORS.get(model, "#f9f9f9")
            for idx, abstract in enumerate(abstracts, 1):
                label = f"{model} — idea {idx}/{len(abstracts)}" if len(abstracts) > 1 else model
                display(HTML(
                    f"""
                    <div style="
                        background:{bg};
                        border:1px solid #d0d0d0;
                        border-left:5px solid #555;
                        border-radius:6px;
                        padding:14px 18px;
                        margin:6px 0 6px 0;
                        font-family:Georgia, serif;
                        line-height:1.7;
                    ">
                        <div style="
                            font-family:monospace; font-size:16px;
                            background:#555; color:#fff;
                            display:inline-block; padding:2px 8px;
                            border-radius:4px; margin-bottom:10px;
                        ">{label}</div>
                        <p style="margin:0; font-size:18px; color:#222;">{abstract}</p>
                    </div>
                    """
                ))

        display(HTML("<hr style='border:none;border-top:1px solid #e0e0e0;margin:16px 0'/>"))


render_results(results, MODELS)